In [1]:
import json
import os

from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from groq import Groq

In [2]:
PROJECT_ROOT = Path.cwd().parent

REVIEWS_PATH = (
    PROJECT_ROOT
    / "data"
    / "intermediate"
    / "reviewed_papers.csv"
)

QUERY_CONFIG_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "arxiv_query.json"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = OUTPUT_DIR / "final_report.json"

MODEL = "llama-3.1-8b-instant"

In [3]:
load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = Groq(
    api_key=GROQ_API_KEY
)

In [4]:
df_reviews = pd.read_csv(REVIEWS_PATH)

In [5]:
with open(
    QUERY_CONFIG_PATH,
    encoding="utf-8",
) as file:
    query_config = json.load(file)

research_topic = query_config["topic"]
arxiv_query = query_config["query"]

print("Research topic:")
print(research_topic)

Research topic:
i want to investigate about the gaussian distribution


In [6]:
df_approved = (
    df_reviews[
        df_reviews["approved"] == True
    ]
    .sort_values(
        "corrected_relevance_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

df_approved[
    [
        "title",
        "corrected_relevance_score",
        "url",
    ]
]

,title,corrected_relevance_score,url
0,A Constructive Approach to $q$-Gaussian Distri...,8,http://arxiv.org/abs/2603.21391v2
1,"Topics in Probability, Parametric Estimation a...",8,http://arxiv.org/abs/2510.20163v3
2,Revisiting De Moivre-Laplace,8,http://arxiv.org/abs/2512.22330v1
3,The fast rate of convergence of the smooth ada...,8,http://arxiv.org/abs/2503.10827v2
4,Probabilistic interpretation of the Selberg--D...,8,http://arxiv.org/abs/2501.17535v1


In [7]:
papers_for_editor = []

for _, paper in df_approved.iterrows():

    papers_for_editor.append(
        {
            "title": paper["title"],
            "authors": paper["authors"],
            "published": str(paper["published"]),
            "url": paper["url"],
            "summary": paper["analyst_summary"],
            "main_problem": paper["main_problem"],
            "main_contribution": paper["main_contribution"],
            "applications": paper["applications"],
            "limitations": paper["limitations"],
            "relevance_score": int(
                paper["corrected_relevance_score"]
            ),
            "relevance_reason": paper[
                "final_relevance_reason"
            ],
        }
    )

In [8]:
papers_text = json.dumps(
    papers_for_editor,
    indent=2,
    ensure_ascii=False,
)

In [9]:
prompt = f"""
You are the Editor Agent in an academic research system.

The user's research topic is:

{research_topic}

You receive a list of papers that have already been analyzed and reviewed.

Your task is to create a concise research briefing.

Reviewed papers:

{papers_text}

Return only a valid JSON object with these fields:

{{
  "executive_summary": "A concise overview of the findings",
  "main_trends": [
    "Trend 1",
    "Trend 2"
  ],
  "key_differences": [
    "Difference 1",
    "Difference 2"
  ],
  "recommended_reading_order": [
    {{
      "position": 1,
      "title": "Paper title",
      "reason": "Why this paper should be read at this position"
    }}
  ],
  "final_recommendation": "A practical recommendation for the user"
}}

Rules:

- Use only the information provided.
- Do not invent facts.
- Mention common patterns across papers.
- Explain important differences between approaches.
- Recommended papers must come from the provided list.
- Keep the response concise.
- Do not use Markdown.
- Return only JSON.
"""

In [10]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    response_format={
        "type": "json_object"
    },
    temperature=0.2,
)

raw_report = response.choices[0].message.content

print(raw_report)

{
  "executive_summary": "The reviewed papers investigate various aspects of the Gaussian distribution, including its relation to power-law distributions, stochastic calculus, and geometric probability.",
   "main_trends": [
      "The application of the Gaussian distribution in different fields, such as physics, finance, and social sciences.",
      "The development of new methods and techniques for understanding and modeling power-law distributions."
   ],
   "key_differences": [
      "The focus on different aspects of the Gaussian distribution, such as its relation to power-law distributions or stochastic calculus.",
      "The use of different mathematical tools and techniques, such as nonlinear differential equations or geometric probability."
   ],
   "recommended_reading_order": [
      {
         "position": 1,
         "title": "A Constructive Approach to $q$-Gaussian Distributions: $α$-Divergence as Rate Function and Generalized de Moivre-Laplace Theorem",
         "reason":

In [11]:
report = json.loads(raw_report)

In [12]:
final_report = {
    "research_topic": research_topic,
    "arxiv_query": arxiv_query,
    "papers_found": len(df_reviews),
    "papers_approved": len(df_approved),
    "report": report,
    "papers": papers_for_editor,
}

In [13]:
with open(
    REPORT_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        final_report,
        file,
        indent=4,
        ensure_ascii=False,
    )

print(f"Final report saved to:\n{REPORT_PATH}")

Final report saved to:
C:\Users\jortialo\ai-paper-review-agentic\data\output\final_report.json


In [14]:
print("EXECUTIVE SUMMARY")
print(report["executive_summary"])

EXECUTIVE SUMMARY
The reviewed papers investigate various aspects of the Gaussian distribution, including its relation to power-law distributions, stochastic calculus, and geometric probability.


In [15]:
print("KEY DIFFERENCES")

for difference in report["key_differences"]:
    print(f"- {difference}")

KEY DIFFERENCES
- The focus on different aspects of the Gaussian distribution, such as its relation to power-law distributions or stochastic calculus.
- The use of different mathematical tools and techniques, such as nonlinear differential equations or geometric probability.


In [16]:
print("RECOMMENDED READING ORDER")

for item in report["recommended_reading_order"]:

    print(
        f'{item["position"]}. '
        f'{item["title"]}'
    )

    print(f'   {item["reason"]}')

RECOMMENDED READING ORDER
1. A Constructive Approach to $q$-Gaussian Distributions: $α$-Divergence as Rate Function and Generalized de Moivre-Laplace Theorem
   This paper provides a comprehensive overview of the q-Gaussian distribution and its relation to power-law distributions.
2. Revisiting De Moivre-Laplace
   This paper provides a new proof of the de Moivre-Laplace theorem, which is closely related to the central limit theorem and the Gaussian distribution.


In [17]:
print("FINAL RECOMMENDATION")
print(report["final_recommendation"])

FINAL RECOMMENDATION
Start by reading the first paper to gain a comprehensive understanding of the q-Gaussian distribution, and then move on to the second paper to gain a deeper understanding of the de Moivre-Laplace theorem and its relation to the Gaussian distribution.
